<a href="https://colab.research.google.com/github/nepslor/teaching/blob/main/TimeSeriesForecasting/W4/kalman_filter_exercise_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kalman filter forecasting
In this exercise we will try to forecast the position of a polecart pendolum using a simplified linear model for the system's dynamics.
Called $x$ the position, $\theta$ the angle of the pendulum, the full equations of motion for the cartpole are:
$$
\begin{align}
x_{t+1} &= x_{t}+\dot{x} dt +\frac{1}{2}\ddot{x} t^2\\
\theta_{t+1} &= \theta_{t}+\dot{\theta} dt +\frac{1}{2}\ddot{\theta} t^2\\
\end{align}
$$

Unfortunately, the expressions for the cartesian and angular accelerations are non-linear function of the state:
$$
\begin{aligned}
\ddot{x} &= \frac{m L \dot{\theta}^2 \sin(\theta) + m g \sin(\theta) \cos(\theta) + u(t) - d \dot{x}}{M + m \sin^2(\theta)}\\
\ddot{\theta} &= \frac{-m L \dot{\theta}^2 \sin(\theta) \cos(\theta) - (M + m) g \sin(\theta) - \cos(\theta) u(t) + d \cos(\theta) \dot{x}}{L (M + m \sin^2(\theta))}
\end{aligned}
$$
where $m$ and  $g$ the mass of the pendulum and the gravitational acceleration, and $u$ an external force acting on the cartpole.

We can approximate the dynamics of the cartpole with a linear system:

$$
\begin{bmatrix}
x(k+1) \\
\dot{x}(k+1) \\
\theta(k+1) \\
\dot{\theta}(k+1)
\end{bmatrix} =
\begin{bmatrix}
1 & dt & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & dt \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
x(k) \\
\dot{x}(k) \\
\theta(k) \\
\dot{\theta}(k)
\end{bmatrix}
$$


❓ What's the interpretation of the linear system above?


# Kalman filter for foreacasting
In this exercise we'll try to write a Kalman filter from scratch and use it to forecast the motion of the cart pole in the next 10 steps.
We report here the main prediction and update formula of the Kalman filter:

**state prediction**
$$\begin{align}
\hat{x}_{k|k-1} &= A_k \hat{x}_{k-1|k-1}\\
P_{k|k-1} &= A_k P_{k-1|k-1} A_k^T + Q_k
\end{align}$$


**state update**
$$\begin{align}
\tilde{y}_k &= y_k - C_k \hat{x}_{k|k-1}\\
S_k &= C_k P_{k|k-1} C_k^T + R_k\\
K_k &= P_{k|k-1} C_k^T S_k^{-1}\\
\hat{x}_{k|k} &= \hat{x}_{k|k-1} + K_k \tilde{y}_k\\
P_{k|k} &= (I - K_k C_k) P_{k|k-1}
\end{align}$$




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from functools import partial
from IPython.display import HTML

The following block defines the nonlinear equation of the true system and simulate them over 500 steps.

In [ ]:
# Pendulum-cart dynamics as provided by user
def pendcart_ode(t, state, ctrl, m=1., M=5., L=2., g=9.81, d=1.):
    x, x_dot, theta, theta_dot = state
    ctrl_t = ctrl(t)
    s, c = np.sin(theta), np.cos(theta)

    x_dot_dot = (m * L * theta_dot ** 2 * s + m * g * s * c + ctrl_t - d * x_dot) / (M + m * s ** 2)
    theta_dot_dot = (-m * L * theta_dot ** 2 * s * c - (M + m) * g * s - c * ctrl_t + d * c * x_dot) / (L * (M + m * s ** 2))

    return [x_dot, x_dot_dot, theta_dot, theta_dot_dot]

# Simulation helper function
def simulate_pendcart(x0, ctrl, t, **kwargs):
    ctrl_func = interp1d(t, ctrl, kind='linear', fill_value="extrapolate")
    sol = solve_ivp(partial(pendcart_ode, ctrl=ctrl_func, **kwargs), [t[0], t[-1]], x0, t_eval=t)
    return sol.y.T

# Define simulation parameters
n_osc = 10
t_end = 20
t = np.linspace(0, t_end, 500)
dt = t[1] - t[0]
initial_state = [0, 0, np.pi / 6, 0]
control_signal = 3* np.sin(n_osc * t * 2 * np.pi /t_end) +  10 * np.random.normal(0, 2, size=t.shape)

# Simulate system dynamics
states = simulate_pendcart(initial_state, control_signal, t)


The following block just define an animation

In [ ]:
#@title Animation
# Create animation
fig, ax = plt.subplots(figsize=(8, 5), layout='tight')
ax.set_xlim(-10, 10)
ax.set_ylim(-3, 3)
# axis equal
ax.set_aspect('equal')
ax.grid()

cart_width, cart_height = 0.4, 0.2
cart_patch = plt.Rectangle((0, 0), cart_width, cart_height, fc='black')
line, = ax.plot([], [], 'o-', lw=2, markersize=8)
ax.add_patch(cart_patch)

def init():
    cart_patch.set_xy((-cart_width / 2, -cart_height / 2))
    line.set_data([], [])
    return cart_patch, line

def animate(i):
    cart_x = states[i, 0]
    theta = states[i, 2]

    cart_patch.set_xy((cart_x - cart_width / 2, -cart_height / 2))
    pend_x = cart_x + 2 * np.sin(theta)
    pend_y = -2 * np.cos(theta)

    line.set_data([cart_x, pend_x], [0, pend_y])
    return cart_patch, line

ani = animation.FuncAnimation(fig, animate, frames=100, interval=40, init_func=init, blit=True);

# Display animation in Colab
HTML(ani.to_jshtml())

# ❓ KF implementation
Try to complete the `kalman_predict` and  `kalman_update` functions using the formula above

In [ ]:
# Kalman filter functions
def kalman_predict(x, P, A, Q):
    x_pred = A @ x
    P_pred = A @ P @ A.T + Q
    return x_pred, P_pred

def kalman_update(x_pred, P_pred, y, C, R):
    S = C @ P_pred @ C.T + R
    K = P_pred @ C.T @ np.linalg.inv(S)
    x_updated = x_pred + K @ (y - C @ x_pred)
    P_updated = (np.eye(len(P_pred)) - K @ C) @ P_pred
    return x_updated, P_updated

def kalman(x, P, A, Q, y, C, R):
    x_pred, P_pred = kalman_predict(x, P, A, Q)
    x_updated, P_updated = kalman_update(x_pred, P_pred, y, C, R)
    return x_updated, P_updated



# ❓Define the system
Try to define matrix $A$ and $C$ for our system

In [ ]:

# Simulate system dynamics
states = simulate_pendcart(initial_state, control_signal, t)

# Kalman Filter parameters
A = np.array([[1, dt, 0, 0],
              [0, 1, 0, 0],
              [0, 0, 1, dt],
              [0, 0, 0, 1]])

Q = 0.01 * np.eye(4)
C = np.eye(4)
R = 0.05 * np.eye(4)
P = np.eye(4)



The following code runs the `kalman_forecast` function at each step. This function calls the KF you designed and runs it for 10 steps, producing an estimatino of the evolution of the system.

❓ Don't we need a forecast of the external force to predict the next states?

In [ ]:
def kalman_forecast(x, P, A, Q, C, steps=10):
    forecasts = []
    x_forecast, P_forecast = x.copy(), P.copy()
    for _ in range(steps):
        x_forecast, P_forecast = kalman_predict(x_forecast, P_forecast, A, Q)
        forecasts.append(C @ x_forecast)
    return np.array(forecasts)

# Prepare Kalman filter forecasts
forecasts = []
x = states[0]
for i in range(len(t)-10):
    y = states[i]
    x_pred, P_pred = kalman_predict(x, P, A, Q)
    x, P = kalman_update(x_pred, P_pred, y, C, R)
    forecast = kalman_forecast(x, P, A, Q, C, steps=10)
    forecasts.append(forecast)

In [ ]:
#@title Forecast animation
# Create animation with Kalman forecasts
fig, ax = plt.subplots(figsize=(10, 5), layout='tight')
ax.set_aspect('equal')
ax.set_xlim(-20, 20)
ax.set_ylim(-3, 3)
ax.grid()

cart_width, cart_height = 0.4, 0.2
cart_patch = plt.Rectangle((0, 0), cart_width, cart_height, fc='black')
line, = ax.plot([], [], 'o-', lw=2, markersize=8)
pred_line, = ax.plot([], [], 'r--', lw=1.5)
ax.add_patch(cart_patch)

def init():
    cart_patch.set_xy((-cart_width / 2, -cart_height / 2))
    line.set_data([], [])
    pred_line.set_data([], [])
    return cart_patch, line, pred_line

def animate(i):
    cart_x = states[i, 0]
    theta = states[i, 2]

    cart_patch.set_xy((cart_x - cart_width / 2, -cart_height / 2))
    pend_x = cart_x + 2 * np.sin(theta)
    pend_y = -2 * np.cos(theta)
    line.set_data([cart_x, pend_x], [0, pend_y])

    if i < len(forecasts):
        pred_cart_x = forecasts[i][:, 0]
        pred_theta = forecasts[i][:, 2]
        pred_pend_x = pred_cart_x + 2 * np.sin(pred_theta)
        pred_pend_y = -2 * np.cos(pred_theta)
        pred_line.set_data(pred_pend_x, pred_pend_y)

    return cart_patch, line, pred_line

ani = animation.FuncAnimation(fig, animate, frames=200, interval=40, init_func=init, blit=True)
# Display animation in Colab
HTML(ani.to_jshtml())